# HF TDOA Analysis - Figure 15

This notebook uses the `hf_tdoa` library to analyze HF TDOA measurements.

In [1]:
import os
import datetime
import importlib
import hf_tdoa as tdoa

%matplotlib inline

# Setup plotting style
tdoa.setup_plotting_style()

## Load WAV Files & Find Chirps

In [ ]:
base_dir   = '../data'
data_set   = 'TX_WA5FRF_EL09nn-RX_AB5YO_EL09so-40m'
sweep_rate = 10  # Hz/ms

# Path to chirp template - used for finding chirp locations via cross-correlation
template   = os.path.join('../templates', 'N6RFM_10Hz_per_ms_template.wav')

data_dir   = os.path.join(base_dir, data_set)
wavlist    = tdoa.obtain_wav_list(data_dir)

In [ ]:
# Correlate each WAV with a known template chirp to identify chirp locations in each WAV file.
chirps = tdoa.find_chirps(wavlist, template, sweep_rate=sweep_rate, plot_correlation=False)

## Find TDOAs

In [ ]:
debug_TDOAs = False

# Find TDOAs for each propagation mode using the simplified API
# The mode configurations (filter limits, search limits, model coefficients, and plotting params)
# are now stored in hf_tdoa_lib.MODE_CONFIGS

# Process all three modes
for mode_string in ['3F2-1F2', '2F2-1F2']:
    chirps = tdoa.find_TDOAs(chirps, mode_string=mode_string,
                            plot_fft=debug_TDOAs, only_one=debug_TDOAs)

# Build the TDOA configuration dictionary with model coefficients and plotting parameters
tdoa_dct = tdoa.build_tdoa_config(chirps)

# Print calculated model coefficients for verification
print("Calculated TDOA Model Coefficients:")
print("=" * 60)
for set_name, params in tdoa_dct.items():
    slope, intercept = params['model_coeffs']
    mode_str = params['mode_string']
    print(f"{set_name:12s} ({mode_str:8s}): slope={slope:6.1f}, intercept={intercept:6.1f}")
print("=" * 60)
print()

## Figure 15 - TDOA Measurements and Layer Heights

This figure combines TDOA measurements (a) and layer heights comparison with ionosonde (b) in a single figure with two subplots.

In [ ]:
tdoa.plot_tdoa_hmf2_subplot(chirps, tdoa_dct, 
                            ylim_tdoa=(0, 5), 
                            ylim_hmf2=(200, 350),
                            ionosonde_dct=True,
                            savefig='fig_15.jpg')

In [ ]:
# Access the path_info object stored when chirps were created
path_info = chirps.attrs['path_info']

# Display path information
print(f"Path Information:")
print(f"  {path_info}")
print()

# Get TX and RX coordinates from gridsquares
tx_lat, tx_lon = path_info.get_tx_latlon()
rx_lat, rx_lon = path_info.get_rx_latlon()
print(f"TX Location: {tx_lat:.3f}°N, {tx_lon:.3f}°E ({path_info.tx_grid})")
print(f"RX Location: {rx_lat:.3f}°N, {rx_lon:.3f}°E ({path_info.rx_grid})")
print()

# Calculate the great circle midpoint between TX and RX
mid_lat, mid_lon = path_info.get_midpoint()
print(f"Path Midpoint: {mid_lat:.3f}°N, {mid_lon:.3f}°E")
print()

# Calculate the azimuth from TX to RX
azimuth = path_info.get_path_azimuth()
print(f"Path Azimuth (TX→RX): {azimuth:.1f}°")
print()